In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [209]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import pandas as pd
from tqdm import tqdm
from PIL import Image

In [113]:
# Setup of the paths

CONTENT_PATH = os.getcwd()
PROJECT_PATH = os.path.join(CONTENT_PATH, 'drive', 'MyDrive', 'AdvancedML', 'Project')

IMAGES_DIR_PATH = os.path.join(PROJECT_PATH, 'release', 'images')
TRAIN_CSV_PATH = os.path.join(PROJECT_PATH,'release','train.csv')
TEST_CSV_PATH = os.path.join(PROJECT_PATH,'release','test_episodes_release.csv')

In [213]:
class DatasetAnalyzer():



  def __init__(self, imgs_dir, train_csv_path, test_csv_path):

    self.imgs_dir = imgs_dir
    self.train_df = pd.read_csv(train_csv_path)
    self.test_df = pd.read_csv(test_csv_path)



  def extract_info(self, df, filename_col, label_col):

    img_paths = []
    labels = []

    for row in df.itertuples():

        filename = getattr(row, filename_col)
        label = getattr(row, label_col)

        img_path = os.path.join(self.imgs_dir, filename)

        img_paths.append(img_path)
        labels.append(label)

    return img_paths, labels



  def get_dataset_info(self):

    train_img_paths, train_labels = self.extract_info(
        self.train_df,
        "filename",
        "label"
    )

    test_img_paths, test_labels = self.extract_info(
        self.test_df,
        "filename",
        "label"
    )

    print(f"TRAIN IMAGES: {len(train_img_paths)}")
    print(f"TRAIN CLASSES: {len(set(train_labels))}")

    print(f"TEST IMAGES: {len(test_img_paths)}")
    print(f"TEST CLASSES: {pd.Series(test_labels).nunique(dropna=True)}")



  def show_train_random_imgs(self):

    train_idxs = np.arange(1,5001,1)
    random_idxs = np.random.randint(1, 5001, size=5)

    fig, axs = plt.subplots(1, 5, figsize=(12,12))
    for i in range(5):
      IMG_PATH = os.path.join(IMAGES_DIR_PATH, os.listdir(IMAGES_DIR_PATH)[random_idxs[i]])
      img = cv.imread(IMG_PATH)
      axs[i].imshow(img)
      axs[i].set_axis_off()


  def inspect_episode(self, episode_idx):

    episode_df = self.test_df[self.test_df['episode_id'] == episode_idx]

    support_set = episode_df[episode_df.role == 'support']
    query_set = episode_df[episode_df.role == 'query']

    classes = support_set.label.unique().astype(int)
    print(f"Episode {episode_idx} has these classe: {classes}")

    fig, axs = plt.subplots(len(classes), len(support_set)//len(classes), figsize=(12,12))
    fig.suptitle(f"Episode {episode_idx} - SUPPORT", fontsize=16, color='green')

    for i, cls in enumerate(classes):
      subset = support_set[support_set.label == cls].reset_index(drop=True)

      for j in range(5):
        filename = subset.iloc[j]['filename']
        file_path = os.path.join(self.imgs_dir,filename)

        img = cv.imread(file_path)
        img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

        ax = axs[i,j]
        ax.imshow(img)
        for spine in ax.spines.values():
            spine.set_visible(True)      # Rendi il bordo visibile
            spine.set_edgecolor('green') # Imposta il colore
            spine.set_linewidth(2)       # Spessore visibile

        ax.set_xticks([])
        ax.set_yticks([])


    fig1, axs1 = plt.subplots(5,5,figsize=(12,12))
    fig1.suptitle(f"Episode {episode_idx} - QUERY", fontsize=16, color='blue')
    axs1 = axs1.flatten()

    for z in range(axs1.shape[0]):
      filename = query_set.iloc[z]['filename']
      file_path = os.path.join(self.imgs_dir,filename)

      img = cv.imread(file_path)
      img = cv.cvtColor(img, cv.COLOR_BGR2RGB)

      ax = axs1[z]
      ax.imshow(img)
      for spine in ax.spines.values():
            spine.set_visible(True)      # Rendi il bordo visibile
            spine.set_edgecolor('blue') # Imposta il colore
            spine.set_linewidth(2)       # Spessore visibile

      ax.set_xticks([])
      ax.set_yticks([])

    plt.tight_layout()
    plt.show()



  def check_data_integrity(self):

    corrupted_files = []
    all_files = pd.concat([self.train_df['filename'], self.test_df['filename']]).unique()

    print("Verifica integrità immagini...")
    for filename in tqdm(all_files):
        img = cv.imread(os.path.join(self.imgs_dir, filename))
        if img is None:
            corrupted_files.append(filename)

    if len(corrupted_files) > 0:
        print(f"ATTENZIONE: {len(corrupted_files)} file corrotti o non leggibili trovati!")
        return corrupted_files
    else:
        print("Tutte le immagini sono leggibili. Dataset integro.")


  def plot_class_distribution(self):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    # Train
    # Sostituisci il .plot(...) precedente con questo
    self.train_df['label'].value_counts().plot(kind='bar', ax=ax[0], color='salmon')
    ax[0].tick_params(axis='x') # Ruota le etichette di 45 gradi
    ax[0].set_xticks([])
    # Test
    self.test_df['label'].value_counts().plot(kind='bar', ax=ax[1], color='skyblue')
    ax[1].set_title('Distribuzione Classi TEST')
    ax[1].set_xticks([])

    plt.tight_layout()
    plt.show()


  def show_imgs_from_class(self, label_id, n_imgs=10):

    print(np.array(sorted(self.train_df.label.unique())).astype(int))
    assert label_id in self.train_df.label.unique(), 'Classe non trovata'
    # Filtra il train set per la label scelta
    class_df = self.train_df[self.train_df['label'] == label_id].head(n_imgs)


    fig, axs = plt.subplots(1, n_imgs, figsize=(n_imgs*2, 2))
    for i, (_, row) in enumerate(class_df.iterrows()):
        img = cv.cvtColor(cv.imread(os.path.join(self.imgs_dir, row['filename'])), cv.COLOR_BGR2RGB)
        axs[i].imshow(img)
        axs[i].set_axis_off()
    plt.show()


  def check_class_overlap(self):

    train_classes = set(self.train_df['label'].unique())
    test_classes = set(self.test_df['label'].unique())

    overlap = train_classes.intersection(test_classes)

    if not overlap:
        print("Ottimo: Non ci sono classi in comune tra Train e Test.")
        print("Il dataset è perfettamente disgiunto.")

    else:
        print(f"Attenzione: Sono state trovate {len(overlap)} classi in comune!")
        print(f"Classi sovrapposte: {sorted(list(overlap))}")
        return overlap


  def check_image_resolutions(self):
    # Uniamo i file per analizzare tutto il dataset
    all_files = pd.concat([self.train_df['filename'], self.test_df['filename']]).unique()

    resolutions = set()
    print("Analisi risoluzioni in corso...")

    for i,filename in enumerate(all_files):
        img_path = os.path.join(self.imgs_dir, filename)
        with Image.open(img_path) as img:
          w, h = img.size
          resolutions.add((w, h))

        if i > 20:
          break


    if len(resolutions) == 1:
        res = list(resolutions)[0]
        print(f"Tutte le immagini hanno la stessa risoluzione: {res[0]}x{res[1]}")
    else:
        print(f"Il dataset ha {len(resolutions)} risoluzioni diverse.")
        print(f"Risoluzioni trovate: {sorted(list(resolutions))}")




analyzer = DatasetAnalyzer(IMAGES_DIR_PATH, TRAIN_CSV_PATH, TEST_CSV_PATH)
analyzer.check_image_resolutions()

Analisi risoluzioni in corso...
Tutte le immagini hanno la stessa risoluzione: 256x256
